<a href="https://colab.research.google.com/github/andrewbeyou88/uXUbYJ34sxDN11/blob/main/microwakeword/notebooks/mww-local.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import platform
import subprocess
import sys

if platform.system() == "Darwin":
    subprocess.run([sys.executable, "-m", "pip", "install",
                   "git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version"])

subprocess.run([sys.executable, "-m", "pip", "install",
               "git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f"])

subprocess.run([sys.executable, "-m", "pip", "install", "-e", "./microWakeWord"])

import os
if not os.path.exists("microWakeWord"):
    subprocess.run(["git", "clone", "https://github.com/kahrendt/microWakeWord"])
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "./microWakeWord"])

subprocess.run([sys.executable, "-m", "pip", "install", "piper-sample-generator"])

In [ ]:
import sys
import os

version = f"{sys.version_info.major}.{sys.version_info.minor}"
# Pe Windows calea e diferita
import site
for sp in site.getsitepackages():
    base = os.path.join(sp, "piper_train")
    vits = os.path.join(base, "vits")
    os.makedirs(vits, exist_ok=True)
    open(os.path.join(base, "__init__.py"), 'w').close()
    open(os.path.join(vits, "__init__.py"), 'w').close()
    open(os.path.join(vits, "commons.py"), 'w').close()
print(f"Fixed for Python {version}")

In [ ]:
import os
import requests

os.makedirs("voices", exist_ok=True)

files = {
    "voices/en_US-lessac-low.onnx": "https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/low/en_US-lessac-low.onnx?download=true",
    "voices/en_US-lessac-low.onnx.json": "https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/low/en_US-lessac-low.onnx.json?download=true"
}

for path, url in files.items():
    if not os.path.exists(path):
        print(f"Downloading {path}...")
        r = requests.get(url, stream=True)
        with open(path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Done: {path}")
    else:
        print(f"Already exists: {path}")

In [ ]:
import os
import sys
import subprocess
from IPython.display import Audio

if not os.path.exists("./piper-sample-generator"):
    subprocess.run(["git", "clone", "https://github.com/rhasspy/piper-sample-generator"])

    r = requests.get("https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt", stream=True)
    os.makedirs("piper-sample-generator/models", exist_ok=True)
    with open("piper-sample-generator/models/en_US-libritts_r-medium.pt", 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

    subprocess.run([sys.executable, "-m", "pip", "install", "torch", "torchaudio", "piper-phonemize-cross==1.2.1"])

    if "piper-sample-generator/" not in sys.path:
        sys.path.append("piper-sample-generator/")

subprocess.run([sys.executable, "-m", "piper_sample_generator", "hey beemo",
               "--model", "voices/en_US-lessac-low.onnx",
               "--max-samples", "1",
               "--output-dir", "generated_samples"])

subprocess.run([sys.executable, "-m", "piper_sample_generator", "hey b-mo",
               "--model", "voices/en_US-lessac-low.onnx",
               "--max-samples", "1",
               "--output-dir", "generated_samples"])

Audio("generated_samples/0.wav", autoplay=True)

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "piper_sample_generator", "hey beemo",
               "--model", "voices/en_US-lessac-low.onnx",
               "--max-samples", "1000",
               "--output-dir", "generated_samples"])

subprocess.run([sys.executable, "-m", "piper_sample_generator", "hey b-mo",
               "--model", "voices/en_US-lessac-low.onnx",
               "--max-samples", "1000",
               "--output-dir", "generated_samples"])

In [ ]:
import os
import shutil
import requests
import scipy.io.wavfile
import scipy.signal
import numpy as np
from pathlib import Path
from tqdm import tqdm
import soundfile as sf
import tarfile

output_dir = "./audioset_16k"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

print("Downloading ambient noise archive...")
url = "http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz"
r = requests.get(url, stream=True)
with open("speech_commands.tar.gz", 'wb') as f:
    for chunk in r.iter_content(chunk_size=8192):
        f.write(chunk)

print("Extracting background noise files...")
os.makedirs("speech_commands_raw", exist_ok=True)
with tarfile.open("speech_commands.tar.gz", "r:gz") as tar:
    for member in tar.getmembers():
        if "_background_noise_" in member.name and member.name.endswith(".wav"):
            member.name = os.path.basename(member.name)
            tar.extract(member, "speech_commands_raw")

bg_files = list(Path("speech_commands_raw").glob("**/*.wav"))
print(f"Found {len(bg_files)} ambient noise files.")

for i, file_path in enumerate(tqdm(bg_files, desc="Processing Ambient Noise")):
    try:
        data, samplerate = sf.read(str(file_path))
        if len(data.shape) > 1:
            data = data.mean(axis=1)
        if samplerate != 16000:
            num_samples = int(len(data) * 16000 / samplerate)
            data = scipy.signal.resample(data, num_samples)
        scipy.io.wavfile.write(os.path.join(output_dir, f"{i}.wav"), 16000, (data * 32767).astype(np.int16))
    except Exception as e:
        print(f"Error at {file_path}: {e}")

shutil.rmtree("speech_commands_raw")
os.remove("speech_commands.tar.gz")
print("Done!")

In [ ]:
import sys
import os
import subprocess

if os.path.exists("micro-wake-word-models"):
    sys.path.append(os.path.abspath("micro-wake-word-models"))
    print("Found micro-wake-word-models!")
elif os.path.exists("micro_wake_word"):
    sys.path.append(os.path.abspath("."))
    print("Local module found!")
else:
    print("Cloning repository...")
    subprocess.run(["git", "clone", "https://github.com/esphome/micro-wake-word-models.git"])
    sys.path.append(os.path.abspath("micro-wake-word-models"))

In [ ]:
import sys
import os
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "audiomentations"])

repo_path = os.path.abspath("micro-wake-word-models")
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print("The environment is ready!")

In [ ]:
import os
import scipy.io.wavfile
import numpy as np
import datasets
from tqdm import tqdm

os.makedirs("mit_rirs", exist_ok=True)
print("Populating mit_rirs folder...")

rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train")
rir_dataset = rir_dataset.cast_column("audio", datasets.Audio(decode=False))

for i, row in enumerate(tqdm(rir_dataset, desc="Saving RIRs")):
    audio_info = row['audio']
    if 'path' in audio_info and audio_info['path'] and os.path.exists(audio_info['path']):
        with open(audio_info['path'], 'rb') as f_in, open(f"mit_rirs/{i}.wav", 'wb') as f_out:
            f_out.write(f_in.read())
    elif 'bytes' in audio_info and audio_info['bytes']:
        with open(f"mit_rirs/{i}.wav", 'wb') as f_out:
            f_out.write(audio_info['bytes'])

print(f"Done! {len(os.listdir('mit_rirs'))} RIR files saved.")

In [ ]:
import audiomentations
from audiomentations import AddGaussianNoise

if not hasattr(audiomentations, "AddColorNoise"):
    if hasattr(audiomentations, "AddGaussianNoise"):
        audiomentations.AddColorNoise = audiomentations.AddGaussianNoise
    elif hasattr(audiomentations.augmentations, "add_color_noise"):
        audiomentations.AddColorNoise = audiomentations.augmentations.add_color_noise.AddColorNoise

class FixedColorNoise(AddGaussianNoise):
    def __init__(self, min_snr_db=-10, max_snr_db=30, p=0.5, **kwargs):
        super().__init__(min_amplitude=0.001, max_amplitude=0.015, p=p)

audiomentations.AddColorNoise = FixedColorNoise
print("AddColorNoise patch applied!")

In [ ]:
import sys
import os

possible_paths = [
    os.path.abspath("microWakeWord"),
    os.path.abspath("micro-wake-word-models"),
    os.path.abspath(".")
]
for p in possible_paths:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

clips = Clips(input_directory='generated_samples',
              file_pattern='*.wav',
              max_clip_duration_s=None,
              remove_silence=False,
              random_split_seed=10,
              split_count=0.1,
              )

augmenter = Augmentation(augmentation_duration_s=3.2,
                         augmentation_probabilities={
                             "SevenBandParametricEQ": 0.1,
                             "TanhDistortion": 0.1,
                             "PitchShift": 0.1,
                             "BandStopFilter": 0.1,
                             "AddColorNoise": 0.1,
                             "AddBackgroundNoise": 0.75,
                             "Gain": 1.0,
                             "RIR": 0.5,
                         },
                         impulse_paths=['mit_rirs'],
                         background_paths=['audioset_16k'],
                         background_min_snr_db=-5,
                         background_max_snr_db=10,
                         min_jitter_s=0.195,
                         max_jitter_s=0.205,
                         )
print("Augmentation setup done!")

In [ ]:
from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip

random_clip = clips.get_random_clip()
augmented_clip = augmenter.augment_clip(random_clip)
save_clip(augmented_clip, 'augmented_clip.wav')

Audio("augmented_clip.wav", autoplay=True)

In [ ]:
import os
from mmap_ninja.ragged import RaggedMmap

output_dir = 'generated_augmented_features'

if not os.path.exists(output_dir):
    os.mkdir(output_dir)

splits = ["training", "validation", "testing"]
for split in splits:
    out_dir = os.path.join(output_dir, split)
    if not os.path.exists(out_dir):
        os.mkdir(out_dir)

    split_name = "train"
    repetition = 2

    spectrograms = SpectrogramGeneration(clips=clips,
                                         augmenter=augmenter,
                                         slide_frames=10,
                                         step_ms=10,
                                         )
    if split == "validation":
        split_name = "validation"
        repetition = 1
    elif split == "testing":
        split_name = "test"
        repetition = 1
        spectrograms = SpectrogramGeneration(clips=clips,
                                             augmenter=augmenter,
                                             slide_frames=1,
                                             step_ms=10,
                                             )

    RaggedMmap.from_generator(
        out_dir=os.path.join(out_dir, 'wakeword_mmap'),
        sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
        batch_size=100,
        verbose=True,
    )
print("Augmented features generated!")

In [ ]:
import os
import shutil
import requests
import zipfile
import time

output_dir = './negative_datasets'

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir)

link_root = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
filenames = ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']

for fname in filenames:
    url = link_root + fname
    zip_path = os.path.join(output_dir, fname)

    print(f"Downloading {fname}...")
    r = requests.get(url, stream=True)
    with open(zip_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

    print(f"Extracting {fname}...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(output_dir)

    os.remove(zip_path)

    folder = fname.replace('.zip', '')
    total = sum(len(files) for _, _, files in os.walk(f"negative_datasets/{folder}"))
    print(f"Done: {fname} — {total} files extracted")
    time.sleep(5)

In [ ]:
import os
import yaml

os.makedirs("D:/mww-local/trained_models/wakeword", exist_ok=True)

config = {}
config["window_step_ms"] = 10
config["train_dir"] = "D:/mww-local/trained_models/wakeword"

config["features"] = [
    {
        "features_dir": "generated_augmented_features",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/speech",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/dinner_party",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/no_speech",
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/dinner_party_eval",
        "sampling_weight": 0.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "split",
        "type": "mmap",
    },
]

config["training_steps"] = [10000]
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]
config["learning_rates"] = [0.001]
config["batch_size"] = 32
config["time_mask_max_size"] = [0]
config["time_mask_count"] = [0]
config["freq_mask_max_size"] = [0]
config["freq_mask_count"] = [0]
config["eval_step_interval"] = 500
config["clip_duration_ms"] = 1500
config["target_minimization"] = 5.0
config["minimization_metric"] = None
config["maximization_metric"] = "average_viable_recall"

config_path = "D:/mww-local/trained_models/wakeword/training_config.yaml"
with open(config_path, "w") as file:
    yaml.dump(config, file)

print(f"Config saved to {config_path}")

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "microwakeword.model_train_eval",
    "--training_config", "D:/mww-local/trained_models/wakeword/training_config.yaml",
    "--train", "1",
    "--restore_checkpoint", "1",
    "--test_tf_nonstreaming", "0",
    "--test_tflite_nonstreaming", "0",
    "--test_tflite_nonstreaming_quantized", "0",
    "--test_tflite_streaming", "0",
    "--test_tflite_streaming_quantized", "1",
    "--use_weights", "best_weights",
    "mixednet",
    "--pointwise_filters", "64,64,64,64",
    "--repeat_in_block", "1, 1, 1, 1",
    "--mixconv_kernel_sizes", "[5], [7,11], [9,15], [23]",
    "--residual_connection", "0,0,0,0",
    "--first_conv_filters", "32",
    "--first_conv_kernel_size", "5",
    "--stride", "3"
])

In [ ]:
import os

tflite_path = "D:/mww-local/trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"

if os.path.exists(tflite_path):
    print(f"Model gata la: {tflite_path}")
else:
    print("Modelul nu a fost gasit, verifica antrenarea.")